In [ ]:
# ===============================
# 1) IMPORTS
# ===============================
import os
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

import timm

print("[INFO] Imports loaded.")


In [ ]:
ROOT = "/kaggle/input/cassava-leaf-disease-classification"
CSV_PATH = os.path.join(ROOT, "train.csv")
IMG_DIR = os.path.join(ROOT, "train_images")

TEST_DIR = "/kaggle/input/cassava-leaf-disease-classification/test_images"
SAMPLE_SUB = os.path.join(ROOT, "sample_submission.csv")

WEIGHT_PATH = "/kaggle/working/best_efficientnet_b3.pth"

print("[INFO] Paths ready.")



In [ ]:
import pandas as pd

df = pd.read_csv(CSV_PATH)

print(df.head())

class_counts = df['label'].value_counts().sort_index()

print("\nNumber of samples per class:")
print(class_counts)

print("\nTotal images:", len(df))


In [ ]:
import os
import random
import cv2
import matplotlib.pyplot as plt

label_map_path = os.path.join(ROOT, "label_num_to_disease_map.json")

import json
with open(label_map_path, "r") as f:
    label_map = json.load(f)

print(label_map)

for label in sorted(df['label'].unique()):
    subset = df[df['label'] == label].sample(1, random_state=42)

    print(f"\nClass {label} — {label_map[str(label)]}")

    for _, row in subset.iterrows():
        img_path = os.path.join(IMG_DIR, row["image_id"])

        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        plt.imshow(img)
        plt.title(f"Class {label} — {label_map[str(label)]}")
        plt.axis("off")
        plt.show()


In [ ]:
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BATCH_SIZE = 64
FREEZE_EPOCHS = 5          # phase 1
FINETUNE_EPOCHS = 30       # phase 2 + 3
LR = 1e-3
VAL_SPLIT = 0.15
AMP = True
EMA_DECAY = 0.999

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)
print(f"[INFO] Using device: {DEVICE}")


In [ ]:
class CassavaDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.values
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image_id, label = self.df[idx]
        img_path = os.path.join(IMG_DIR, image_id)

        image = np.array(Image.open(img_path).convert("RGB"))

        if self.transform:
            image = self.transform(image=image)["image"]

        return image, torch.tensor(label, dtype=torch.long)


print("[INFO] Dataset class ready.")



In [ ]:

# train_tf = A.Compose([
#     A.RandomResizedCrop(
#         size=(300, 300),
#         scale=(0.9, 1.0),
#         ratio=(0.95, 1.05),
#         p=1.0
#     ),

#     A.HorizontalFlip(p=0.4),

#     A.RandomBrightnessContrast(
#         brightness_limit=0.12,
#         contrast_limit=0.12,
#         p=0.4
#     ),

#     A.Affine(
#         scale=(0.98, 1.02),
#         translate_percent=(0.0, 0.03),
#         rotate=(-7, 7),
#         p=0.3
#     ),

#     A.GaussianBlur(blur_limit=(3, 3), p=0.15),

#     A.Normalize(),
#     ToTensorV2()
# ])

train_tf = A.Compose([
    A.RandomResizedCrop(size=(300, 300), scale=(0.8, 1.0), p=1.0),

    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.15),

    A.ColorJitter(0.2, 0.2, 0.15, 0.02, p=0.7),

    A.GaussNoise(var_limit=(10, 25), p=0.5),
    A.Sharpen(alpha=(0.1, 0.4), p=0.4),

    A.CoarseDropout(
        max_holes=12,
        max_height=50,
        max_width=50,
        p=0.6
    ),

    A.Normalize(),
    ToTensorV2(),
])

val_tf = A.Compose([
    A.Resize(300, 300),
    A.Normalize(),
    ToTensorV2()
])

def cutmix(imgs, labels, alpha=1.0):
    lam = np.random.beta(alpha, alpha)
    batch = imgs.size(0)

    idx = torch.randperm(batch)

    bbx1 = np.random.randint(0, imgs.size(2))
    bby1 = np.random.randint(0, imgs.size(3))

    cut_w = int(imgs.size(2) * np.sqrt(1 - lam))
    cut_h = int(imgs.size(3) * np.sqrt(1 - lam))

    x1 = np.clip(bbx1 - cut_w // 2, 0, imgs.size(2))
    y1 = np.clip(bby1 - cut_h // 2, 0, imgs.size(3))
    x2 = np.clip(bbx1 + cut_w // 2, 0, imgs.size(2))
    y2 = np.clip(bby1 + cut_h // 2, 0, imgs.size(3))

    imgs[:, :, x1:x2, y1:y2] = imgs[idx, :, x1:x2, y1:y2]
    lam = 1 - ((x2 - x1) * (y2 - y1) / (imgs.size(-1) * imgs.size(-2)))

    return imgs, labels, labels[idx], lam



In [ ]:
df = pd.read_csv(CSV_PATH)

from sklearn.model_selection import StratifiedShuffleSplit

splitter = StratifiedShuffleSplit(
    n_splits=1,
    test_size=VAL_SPLIT,
    random_state=42
)

for train_idx, val_idx in splitter.split(df, df["label"]):
    train_df = df.iloc[train_idx]
    val_df = df.iloc[val_idx]

print(train_df["label"].value_counts())
print(val_df["label"].value_counts())

train_dataset = CassavaDataset(train_df, train_tf)
val_dataset   = CassavaDataset(val_df, val_tf)


In [ ]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=False)

val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=4, pin_memory=False)

print("[INFO] DataLoaders ready.")


In [ ]:
import timm
import torch
import torch.nn as nn
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
AMP = True
LR = 1e-3

model = timm.create_model(
    "seresnext50_32x4d",
    pretrained=True,
    num_classes=5
).to(DEVICE)

def init_head(m):
    if isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
        if m.bias is not None:
            nn.init.zeros_(m.bias)

model.get_classifier().apply(init_head)

# ---------- FREEZE BACKBONE (phase 1) ----------
for name, p in model.named_parameters():
    if "fc" not in name and "classifier" not in name:
        p.requires_grad = False


# ===============================
# 2) CLASS WEIGHTS
# ===============================

labels = df["label"].values

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(labels),
    y=labels
)

class_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=0.05
)

# ===============================
# 3) OPTIMIZER + SCHEDULER
# ===============================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=1e-4
)

cosine = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    T_0=5,
)

plateau = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    patience=2,
    factor=0.3
)

scaler = torch.cuda.amp.GradScaler(enabled=AMP)

# ---------- EMA ----------
import copy

ema_model = copy.deepcopy(model)
for p in ema_model.parameters():
    p.requires_grad = False

def update_ema():
    with torch.no_grad():
        for p, p_ema in zip(model.parameters(), ema_model.parameters()):
            p_ema.data.mul_(EMA_DECAY).add_(p.data * (1 - EMA_DECAY))

print("[INFO] Training setup ready.")


In [ ]:

def run_epoch(loader, training=True, use_cutmix=False):
    total_loss, correct, total = 0, 0, 0

    model.train() if training else model.eval()

    pbar = tqdm(loader)

    with torch.set_grad_enabled(training):
        for imgs, labels in pbar:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

            if training:
                optimizer.zero_grad()

            if training and use_cutmix and np.random.rand() < 0.5:
                imgs, y1, y2, lam = cutmix(imgs, labels)
                with torch.cuda.amp.autocast(enabled=AMP):
                    outputs = model(imgs)
                    loss = lam * criterion(outputs, y1) + (1 - lam) * criterion(outputs, y2)
            else:
                with torch.cuda.amp.autocast(enabled=AMP):
                    outputs = model(imgs)
                    loss = criterion(outputs, labels)

            if training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                update_ema()

            total_loss += loss.item() * imgs.size(0)
            preds = outputs.argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return total_loss/total, correct/total

print("[INFO] Defining 1 training epoch")


In [ ]:
history = {
    "train_loss": [], "val_loss": [],
    "train_acc":  [], "val_acc": []
}

best_acc = 0


# ============================
# PHASE 1 — FREEZE BACKBONE
# ============================
print("\n====== PHASE 1: FREEZE BACKBONE ======\n")

for epoch in range(FREEZE_EPOCHS):

    tr_loss, tr_acc = run_epoch(train_loader, True, use_cutmix=True)
    va_loss, va_acc = run_epoch(val_loader, False)

    cosine.step()

    history["train_loss"].append(tr_loss)
    history["val_loss"].append(va_loss)
    history["train_acc"].append(tr_acc)
    history["val_acc"].append(va_acc)

    print(f"[P1][Epoch {epoch+1}/{FREEZE_EPOCHS}] "
          f"Train: loss={tr_loss:.4f}, acc={tr_acc:.4f} | "
          f"Val: loss={va_loss:.4f}, acc={va_acc:.4f}")

    if va_acc > best_acc:
        best_acc = va_acc
        torch.save(ema_model.state_dict(), WEIGHT_PATH)
        print(f"New best (P1) — acc={best_acc:.4f}")


# ===============================
# PHASE 2/3 — UNFREEZE + FINETUNE
# ===============================
print("\n====== PHASE 2/3: UNFREEZE & FINETUNE ======\n")

for p in model.parameters():
    p.requires_grad = True

optimizer.param_groups[0]["lr"] = 1e-4


for epoch in range(FINETUNE_EPOCHS):

    tr_loss, tr_acc = run_epoch(train_loader, True, use_cutmix=True)
    va_loss, va_acc = run_epoch(val_loader, False)

    plateau.step(va_loss)

    history["train_loss"].append(tr_loss)
    history["val_loss"].append(va_loss)
    history["train_acc"].append(tr_acc)
    history["val_acc"].append(va_acc)

    print(f"[P2][Epoch {epoch+1}/{FINETUNE_EPOCHS}] "
          f"Train: loss={tr_loss:.4f}, acc={tr_acc:.4f} | "
          f"Val: loss={va_loss:.4f}, acc={va_acc:.4f}")

    if va_acc > best_acc:
        best_acc = va_acc
        torch.save(ema_model.state_dict(), WEIGHT_PATH)
        print(f"New best (P2) — acc={best_acc:.4f}")


print(f"\nDone — BEST VAL ACC = {best_acc:.4f}")


In [ ]:
plt.figure(figsize=(12,4))

# ---- Loss ----
plt.subplot(1,2,1)
plt.plot(history["train_loss"], label="Train loss")
plt.plot(history["val_loss"], label="Val loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("Loss")

# ---- Accuracy ----
plt.subplot(1,2,2)
plt.plot(history["train_acc"], label="Train acc")
plt.plot(history["val_acc"], label="Val acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.title("Accuracy")

plt.show()


In [ ]:
print(f"\n[INFO] Loading weights: {WEIGHT_PATH}")

model.load_state_dict(
    torch.load(WEIGHT_PATH, map_location=DEVICE)
)
model.eval()

print("[INFO] Model loaded successfully!")


In [ ]:
class CassavaTestDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image_id = self.df.iloc[idx].image_id
        path = os.path.join(TEST_DIR, image_id)

        image = np.array(Image.open(path).convert("RGB"))

        if self.transform:
            image = self.transform(image=image)["image"]

        return image_id, image


test_tf = A.Compose([
    A.Resize(300, 300),
    A.Normalize(),
    ToTensorV2()
])

sample_sub = pd.read_csv(SAMPLE_SUB)
test_dataset = CassavaTestDataset(sample_sub, test_tf)

test_loader = DataLoader(test_dataset, batch_size=64,
                         shuffle=False, num_workers=2)

preds = []



In [ ]:
tta_tf = [
    A.Compose([A.Resize(300,300), A.Normalize(), ToTensorV2()]),
    A.Compose([A.Resize(300,300), A.HorizontalFlip(p=1), A.Normalize(), ToTensorV2()])
]

def tta_predict(img):
    preds = []
    with torch.no_grad():
        for tf in tta_tf:
            x = tf(image=img)["image"].unsqueeze(0).to(DEVICE)
            preds.append(model(x))
    return torch.mean(torch.stack(preds), dim=0)
    
with torch.no_grad():
    for image_ids, images in tqdm(test_loader):
        batch_preds = []
        for img in images:
            img_np = img.permute(1,2,0).cpu().numpy()
            p = tta_predict(img_np)
            batch_preds.append(p)

        batch_preds = torch.cat(batch_preds, dim=0)
        labels = batch_preds.argmax(1).cpu().numpy()

        for img, lab in zip(image_ids, labels):
            preds.append((img, int(lab)))





In [ ]:
submission = pd.DataFrame(preds, columns=["image_id", "label"])
submission.to_csv("/kaggle/working/submission.csv", index=False)

print("\nDONE — saved -> /kaggle/working/submission.csv")
